# WinLOLBIN × VRAVEN

Run top → bottom. One feature per cell.

Core figures **F01–F19**, dataset **D00–D03**.

DarkSide: `winlolbin_vraven_darkside.ipynb`

`Models/winlolbin_mlp/` · benign=`0` · malware=`1`


## 0 — Install into this kernel

If packages install, **restart kernel**, then continue from cell 1.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

VRAVEN_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "vraven").is_dir()),
    None,
)
if VRAVEN_ROOT is None:
    raise RuntimeError("Start Jupyter from the root of a VRAVEN source checkout")
assert VRAVEN_ROOT.is_dir(), f"Missing VRAVEN root: {VRAVEN_ROOT}"

def _missing(name: str) -> bool:
    return importlib.util.find_spec(name) is None

needs = []
if _missing("vraven"):
    needs.append(("vraven", ["-e", str(VRAVEN_ROOT)]))
if _missing("torch"):
    needs.append(("torch", ["torch"]))
if _missing("plotly"):
    needs.append(("plotly", ["plotly"]))
if _missing("sklearn"):
    needs.append(("scikit-learn", ["scikit-learn"]))

if needs:
    print("Installing:", [n for n, _ in needs])
    for name, args in needs:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
        print("$", " ".join(cmd))
        subprocess.check_call(cmd)
    print("Restart the kernel, then run from cell 1.")
else:
    print("Kernel OK.")
    print(sys.executable)

## 1 — Paths + imports

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import torch

VRAVEN_ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "vraven").is_dir()),
    None,
)
if VRAVEN_ROOT is None:
    raise RuntimeError("Start Jupyter from the root of a VRAVEN source checkout")
EXAMPLE = VRAVEN_ROOT / "examples" / "windows_malware_mlp"
MODEL_DIR = VRAVEN_ROOT / "Models" / "winlolbin_mlp"
OUTPUT_DIR = VRAVEN_ROOT / "vraven_outputs" / "lolbin"
SRC = VRAVEN_ROOT / "src"

for path in (EXAMPLE, MODEL_DIR, SRC):
    assert path.exists(), f"Missing: {path}"

# Prefer the local checkout over any older installed wheel.
for path in (str(SRC), str(EXAMPLE)):
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

# Drop stale vraven modules if this kernel ran before the DarkSide merge.
for name in list(sys.modules):
    if name == "vraven" or name.startswith("vraven."):
        del sys.modules[name]

from features import CLASS_NAMES, DISPLAY_NAMES
from model import MalwareMLP

import vraven
from vraven import VRavenTheme

# Sanity check for DarkSide compatibility shim
from vraven.reasoning.fingerprint import shared_reasoning_fingerprints  # noqa: F401

VRavenTheme.apply()

TARGET = 1
N_SAMPLES = 32
PLOT_MODE = "interactive_dark"
MODE = "causal"

print("vraven", getattr(vraven, "__version__", "?"))
print("python", sys.executable)
print("vraven file", Path(vraven.__file__).resolve())
print("model", MODEL_DIR)


## 2 — Load model

In [ ]:
model_path = MODEL_DIR / "model.pt"
sample_path = MODEL_DIR / "explain_sample.npz"
assert model_path.is_file(), model_path
assert sample_path.is_file(), sample_path

payload = torch.load(model_path, map_location="cpu", weights_only=True)
arch = payload["architecture"]
feature_names = list(payload["feature_names"])
display_names = [DISPLAY_NAMES.get(n, n) for n in feature_names]

model = MalwareMLP(
    input_dim=int(arch["input_dim"]),
    hidden_sizes=tuple(arch["hidden_sizes"]),
    num_classes=int(arch["num_classes"]),
)
model.load_state_dict(payload["model_state"])
model.eval()
print("architecture", arch)

## 3 — Load explain samples

In [ ]:
sample = np.load(MODEL_DIR / "explain_sample.npz")
features_all = sample["features_scaled"].astype(np.float32)
labels_all = sample["labels"].astype(int)
n = min(N_SAMPLES, len(labels_all))

idx0 = np.flatnonzero(labels_all == 0)[: n // 2]
idx1 = np.flatnonzero(labels_all == 1)[: n - len(idx0)]
keep = np.concatenate([idx0, idx1]) if len(idx0) and len(idx1) else np.arange(n)

features = features_all[keep]
labels = labels_all[keep]
label_names = [CLASS_NAMES[int(v)] for v in labels]
x = torch.from_numpy(features)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=-1).cpu().numpy()

print("batch", tuple(x.shape))
print("benign", int((labels == 0).sum()), "malware", int((labels == 1).sum()))
print("agreement", float((probs.argmax(1) == labels).mean()))

## 4 — Build VRAVEN trace (`causal`)

In [ ]:
report = vraven.explain(
    model,
    x,
    target=TARGET,
    mode=MODE,
    class_names=CLASS_NAMES,
    metadata={
        "task": "windows_lolbin_malware_binary",
        "dataset": "WinLOLBIN-GT",
        "model_dir": str(MODEL_DIR),
    },
)
print(report.summary())
print("enabled", sorted(k for k, v in report.capabilities().items() if v))

## 5 — Attach LRP

In [ ]:
report.lrp_result = vraven.explain_lrp(model, x, target=TARGET)
print("LRP attached", report.lrp_result is not None)

## 6 — Attach perturbation stability

In [ ]:
report.perturbation_result = vraven.run_perturbation_stability(
    model, x, target=TARGET, repeats=4, random_seed=17
)
print("perturbation attached", report.perturbation_result is not None)

## 7 — Attach counterfactual

In [ ]:
focus = int(np.argmax(probs[:, 1]))
report.counterfactual_result = vraven.generate_counterfactual(
    model,
    x[focus : focus + 1].clone(),
    target_class=0,
    steps=120,
    learning_rate=0.05,
    l2_weight=0.01,
    top_changes=15,
)
print("focus", focus, label_names[focus], "success", report.counterfactual_result.success)

## 8 — Build per-sample attribution matrix (for heatmap)

In [ ]:
rows = []
model.eval()
for i in range(x.shape[0]):
    sample_i = x[i : i + 1].detach().clone().requires_grad_(True)
    score = model(sample_i)[0, TARGET]
    grad = torch.autograd.grad(score, sample_i, retain_graph=False)[0]
    rows.append((sample_i.detach() * grad.detach()).cpu().numpy().reshape(-1))
attribution_matrix = np.asarray(rows, dtype=np.float64)
print("attribution_matrix", attribution_matrix.shape)

## 9 — Build consistency vectors (input×grad vs LRP)

In [ ]:
grad_vec = np.abs(
    np.asarray(report.trace.inputs.attribution_values, dtype=np.float64)
).reshape(-1)
consistency_explanations = {"input_times_gradient": grad_vec.tolist()}

lrp_feature = np.zeros(grad_vec.size, dtype=np.float64)
lrp_count = 0
for i in range(min(16, x.shape[0])):
    single = vraven.explain_lrp(model, x[i : i + 1], target=TARGET)
    if single.input_relevance is None:
        continue
    vec = np.abs(np.asarray(single.input_relevance, dtype=np.float64)).reshape(-1)
    if vec.size != grad_vec.size:
        continue
    lrp_feature += vec
    lrp_count += 1

assert lrp_count > 0, "Could not build LRP feature vector"
consistency_explanations["lrp_epsilon"] = (lrp_feature / lrp_count).tolist()
print("consistency methods", list(consistency_explanations), "n_lrp", lrp_count)

---
# Feature figures — one block each

## F01 — Feature Importance (`input_attribution`)

In [ ]:
report.plot(
    "feature_importance",
    feature_names=display_names,
    top_k=25,
    mode=PLOT_MODE,
).show()

## F02 — Feature Attribution Heatmap (`feature_sequence_heatmap`)

In [ ]:
report.plot(
    "feature_heatmap",
    matrix=attribution_matrix,
    feature_names=display_names,
    sample_labels=label_names,
    top_features=18,
    mode=PLOT_MODE,
).show()

## F03 — Contribution Sankey (`decision_sankey`)

In [ ]:
report.plot(
    "contribution",
    feature_names=display_names,
    class_names=CLASS_NAMES,
    top_features=8,
    depth_stages=5,
    mode=PLOT_MODE,
).show()

## F04 — Decision Flow (`decision_flow`)

In [ ]:
report.plot("decision_flow", mode=PLOT_MODE).show()

## F05 — Neuron Impact / Layer-wise Relevance (`neuron_impact`)

In [ ]:
report.plot("layer_relevance", mode=PLOT_MODE).show()

## F06 — Neuron Importance (`neuron_importance`)

In [ ]:
report.plot("neuron_importance", top_k=14, mode=PLOT_MODE).show()

## F07 — Representation Landscape (`latent_flight`, t-SNE)

In [ ]:
report.plot(
    "representation",
    labels=labels.tolist(),
    method="tsne",
    max_layers=4,
    random_seed=17,
    perplexity=min(12.0, max(5.0, len(labels) / 4.0)),
    mode=PLOT_MODE,
).show()

## F08 — Layer-wise Relevance Propagation (`relevance_flow` / LRP)

In [ ]:
report.plot("lrp", max_layers=6, mode=PLOT_MODE).show()

## F09 — Counterfactual Explanation (`counterfactual_delta`)

In [ ]:
report.plot(
    "counterfactual",
    feature_names=display_names,
    class_names=CLASS_NAMES,
    mode=PLOT_MODE,
).show()

## F10 — Perturbation Stability (`perturbation_stability`)

In [ ]:
report.plot("perturbation", mode=PLOT_MODE).show()

## F11 — Explanation Consistency Matrix (`consistency_matrix`)

In [ ]:
report.plot(
    "consistency_matrix",
    explanations=consistency_explanations,
    metric="rank_correlation",
    mode=PLOT_MODE,
).show()

## F12 — Reasoning DNA (`reasoning_dna`)

In [ ]:
report.plot("reasoning_dna", mode=PLOT_MODE).show()

## F13 — Spectral Trace (`spectral_trace`)

In [ ]:
report.plot("spectral_trace", mode=PLOT_MODE).show()

## F14 — Conflict Field (`conflict_field`)

In [ ]:
report.plot("conflict_field", mode=PLOT_MODE).show()

## F15 — Evidence Trajectory (`evidence_trajectory`)

In [ ]:
report.plot("evidence_trajectory", mode=PLOT_MODE).show()

## F16 — Causal Pathway (`causal_pathway`)

In [ ]:
report.plot("causal_pathway", mode=PLOT_MODE).show()

## F17 — Activation Energy (`activation_energy`)

In [ ]:
report.plot("activation_energy", mode=PLOT_MODE).show()

## F18 — Reasoning Graph (`reasoning_graph`)

In [ ]:
report.plot("reasoning_graph", mode=PLOT_MODE).show()

## F19 — Activation Community Graph (`community_graph`)

In [ ]:
report.plot(
    "community",
    n_clusters=8,
    similarity_threshold=0.12,
    mode=PLOT_MODE,
).show()

---
# Dataset-level features — one block each

## D00 — Build dataset report

In [ ]:
dataset_report = vraven.explain_dataset(
    model,
    [
        vraven.DatasetItem(
            inputs=x[i : i + 1],
            target=TARGET,
            label=int(labels[i]),
            sample_id=f"{label_names[i]}_{i}",
        )
        for i in range(len(labels))
    ],
    mode="gradient",
    class_names=CLASS_NAMES,
    max_samples=len(labels),
    fingerprint_dimensions=128,
)
print("dataset samples", len(labels))

## D01 — Dataset Reasoning Landscape (`reasoning_landscape`, t-SNE)

In [ ]:
highlight = int(np.argmax(probs[:, 1]))
dataset_report.reasoning_landscape(
    colour_by="label",
    method="tsne",
    highlight_index=highlight,
    random_seed=17,
    perplexity=min(12.0, max(5.0, len(labels) / 4.0)),
    mode=PLOT_MODE,
).show()

## D02 — Dataset Reasoning Similarity Matrix (`reasoning_similarity`)

In [ ]:
dataset_report.reasoning_similarity(mode=PLOT_MODE).show()

## D03 — Dataset Reasoning Outliers (`reasoning_outliers`)

In [ ]:
dataset_report.reasoning_outliers(neighbours=5, mode=PLOT_MODE).show()